# 3D Gaussian splatting in the Jupyter web visualizer

This notebook shows how to load a **3D Gaussian splatting** scene (``.splat``),
inspect the tensor point cloud, view it with the **WebRTC Jupyter** helper
``open3d.web_visualizer.draw``, crop and transform the splats, and combine them
with a **triangle mesh** in the same viewer.

**Requirements**

- A Python build of Open3D with the Jupyter extension (``-DBUILD_JUPYTER_EXTENSION=ON``).
- A network connection for the first run to download the sample scene from Hugging Face
  ([`truck.splat`](https://huggingface.co/cakewalk/splat-data/blob/main/truck.splat)).

The splat file is read with ``open3d.t.io.read_point_cloud``; Gaussian attributes
(positions, scales, rotations, SH coefficients, opacity) follow the usual 3DGS layout.
Use ``PointCloud.Translate`` / ``Rotate`` / ``Scale`` for splats so orientations and
spherical-harmonic bands stay consistent (see the C++ docs for ``t::geometry::PointCloud``).

In [ ]:
import pathlib
import urllib.request

import numpy as np
import open3d as o3d
#from open3d.web_visualizer import draw
from open3d.visualization import draw

# Sample 3DGS scene (Hugging Face — raw file URL for download).
SPLAT_URL = "https://huggingface.co/cakewalk/splat-data/resolve/main/truck.splat"
splat_path = pathlib.Path("truck.splat")

if not splat_path.is_file():
    print(f"Downloading {SPLAT_URL!r} ...")
    urllib.request.urlretrieve(SPLAT_URL, splat_path)
print(f"Using splat file: {splat_path.resolve()}")

## Load the splat scene and print it

Tensor IO loads ``.splat`` / Gaussian ``.ply`` into ``open3d.t.geometry.PointCloud``.
Printing the object summarizes point count and attribute names (e.g. ``positions``,
``scale``, ``rot``, ``opacity``, ``f_dc``, ``f_rest`` when present).

In [ ]:
pcd = o3d.t.io.read_point_cloud(str(splat_path))
print(pcd)

## Display with the web visualizer

``draw`` from ``open3d.web_visualizer`` is the non-blocking Jupyter entry point
(see the **Web visualizer and Jupyter** page in this documentation section).

In [ ]:
draw(pcd, title="Truck (full)", width=800, height=600)

## Crop to the central region

Shrink the axis-aligned bounding box around the cloud centroid so the truck
fills the view and background splats are removed.

In [ ]:
full_bb = pcd.get_axis_aligned_bounding_box()
mn = full_bb.min_bound
mx = full_bb.max_bound
center = (mn + mx) * 0.5
extent = mx - mn
# Keep the middle 45% of each axis (adjust factor as needed).
half = extent * 0.225
crop_bb = o3d.t.geometry.AxisAlignedBoundingBox(center - half, center + half)
pcd_cropped = pcd.crop(crop_bb)
print("Points after crop:", pcd_cropped.point.positions.shape[0])

In [ ]:
draw(pcd_cropped, title="Truck (cropped)", width=800, height=600)

## Translate, rotate, and scale

For Gaussian splat clouds, prefer ``Translate`` / ``Rotate`` / ``Scale`` instead of
a general ``Transform`` so splat orientations and SH coefficients stay valid.

In [ ]:
# Move the cloud so its centroid is at the origin. With relative=False, the
# argument is the desired new center (see t::geometry::PointCloud::Translate).
pcd_xform = pcd_cropped.translate(
    o3d.core.Tensor([0.0, 0.0, 0.0]), relative=False
)

angle = np.pi / 8.0
R = np.array(
    [
        [np.cos(angle), 0.0, np.sin(angle)],
        [0.0, 1.0, 0.0],
        [-np.sin(angle), 0.0, np.cos(angle)],
    ],
    dtype=np.float64,
)
pcd_xform = pcd_xform.rotate(o3d.core.Tensor(R), o3d.core.Tensor([0.0, 0.0, 0.0]))
pcd_xform = pcd_xform.scale(1.15, o3d.core.Tensor([0.0, 0.0, 0.0]))

In [ ]:
draw(pcd_xform, title="Truck (transformed)", width=800, height=600)

## Add a triangle mesh next to the splats

Open3D's ``EaglePointCloud`` dataset is a **colored point cloud** in PLY form, not a
triangle mesh. For a shaded mesh alongside splats, this example uses the Stanford
**bunny** (``open3d.data.BunnyMesh``), scales it, and translates it beside the truck.

Pass a **list** of geometries to ``draw`` to render both together.

In [ ]:
bunny_data = o3d.data.BunnyMesh()
mesh = o3d.t.io.read_triangle_mesh(bunny_data.path)
mesh.compute_vertex_normals()

m_bb = mesh.get_axis_aligned_bounding_box()
m_ctr = m_bb.get_center()
mesh = mesh.scale(0.08, m_ctr)
mesh = mesh.translate(o3d.core.Tensor([2.5, 0.0, 0.0]))

draw([pcd_xform, mesh], title="Truck splats + bunny mesh", width=900, height=650)